データ生成

In [2]:
import pandas as pd

url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/qm9.csv"
df = pd.read_csv(url)

# 列名の自動検出（大文字小文字を無視）
smiles_col = next(c for c in df.columns if "smiles" in c.lower())
gap_col = next(c for c in df.columns if c.lower() == "gap")

HARTREE_TO_EV = 27.211386245988

gap_values = df[gap_col]
# Hartree（値が数十以下）とeV（既に変換済み・数十以上）を値のスケールで自動判定
if gap_values.mean() < 5:
    gap_ev = gap_values * HARTREE_TO_EV
else:
    gap_ev = gap_values

result = pd.DataFrame({
    "smiles": df[smiles_col],
    "gap_eV": gap_ev,
})

result.to_csv("input_data/qm9_smiles_gap_eV.csv", index=False)
print(result.head())
print(f"件数: {len(result)}")
print(len(result) if gap_values.mean() < 5 else None)

  smiles     gap_eV
0      C  13.736308
1      N   9.249150
2      O   9.836916
3    C#C   9.118536
4    C#N  10.329442
件数: 133885
133885


インポート・設定

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator
from rdkit.ML.Descriptors import MoleculeDescriptors

# RDKitの警告ログ（Sanitize失敗などの詳細ログ）を抑制したい場合はコメントを外す
# RDLogger.DisableLog("rdApp.*")

INPUT_CSV = Path("input_data/qm9_smiles_gap_eV.csv")
OUTPUT_DESCRIPTORS_CSV = Path("input_data/qm9_descriptors.csv")
OUTPUT_MORGAN_CSV = Path("input_data/qm9_morgan_fp.csv")
OUTPUT_MACCS_CSV = Path("input_data/qm9_maccs_keys.csv")

MORGAN_RADIUS = 2
MORGAN_N_BITS = 2048

SMILES_COL = "smiles"
LABEL_COL = "gap_eV"

CSV読み込み

In [4]:
df = pd.read_csv(INPUT_CSV)
if SMILES_COL not in df.columns:
    raise ValueError(f"'{SMILES_COL}' 列が見つかりません。実際の列名: {df.columns.tolist()}")

print(f"読み込み件数: {len(df)}")
df.head()

読み込み件数: 133885


,smiles,gap_eV
0,C,13.736308
1,N,9.249150
2,O,9.836916
3,C#C,9.118536
4,C#N,10.329442


SMILES → Mol変換

In [5]:
mols, valid_idx = [], []
for i, smi in enumerate(df[SMILES_COL].tolist()):
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        mols.append(mol)
        valid_idx.append(i)

meta = df.iloc[valid_idx].reset_index(drop=True)
print(f"Mol生成成功: {len(mols)} / 失敗: {len(df) - len(mols)}")

Mol生成成功: 133885 / 失敗: 0


RDKit記述子を計算

In [6]:
descriptor_names = [name for name, _ in Descriptors._descList]
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)

rows = [calculator.CalcDescriptors(mol) for mol in mols]
desc_df = pd.DataFrame(rows, columns=descriptor_names)

# NaN/Infの補完
desc_df = desc_df.replace([np.inf, -np.inf], np.nan)
desc_df = desc_df.fillna(desc_df.median(numeric_only=True))

desc_df = pd.concat([meta[[SMILES_COL, LABEL_COL]], desc_df], axis=1)
desc_df.to_csv(OUTPUT_DESCRIPTORS_CSV, index=False)

print(desc_df.shape)
desc_df.head()

(133885, 219)


,smiles,gap_eV,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,C,13.736308,0.0,0.0,0.0,0.0,0.359785,0.0,16.043,12.011,...,0,0,0,0,0,0,0,0,0,0
1,N,9.249150,0.0,0.0,0.0,0.0,0.397555,0.0,17.031,14.007,...,0,0,0,0,0,0,0,0,0,0
2,O,9.836916,0.0,0.0,0.0,0.0,0.327748,0.0,18.015,15.999,...,0,0,0,0,0,0,0,0,0,0
3,C#C,9.118536,4.0,4.0,4.0,4.0,0.332926,1.0,26.038,24.022,...,0,0,0,1,0,0,0,0,0,0
4,C#N,10.329442,6.5,6.5,3.5,3.5,0.369797,1.0,27.026,26.018,...,0,0,0,0,0,0,0,0,0,0


Morganフィンガープリントを計算

In [8]:

generator = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_N_BITS)

rows = [generator.GetFingerprintAsNumPy(mol) for mol in mols]
morgan_df = pd.DataFrame(rows, columns=[f"morgan_{i}" for i in range(MORGAN_N_BITS)])

morgan_df = pd.concat([meta[[SMILES_COL, LABEL_COL]], morgan_df], axis=1)
morgan_df.to_csv(OUTPUT_MORGAN_CSV, index=False)

print(morgan_df.shape)
morgan_df.head()

(133885, 2050)


,smiles,gap_eV,morgan_0,morgan_1,morgan_2,morgan_3,morgan_4,morgan_5,morgan_6,morgan_7,...,morgan_2038,morgan_2039,morgan_2040,morgan_2041,morgan_2042,morgan_2043,morgan_2044,morgan_2045,morgan_2046,morgan_2047
0,C,13.736308,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,N,9.249150,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,O,9.836916,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,C#C,9.118536,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C#N,10.329442,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


MACCSキーを計算

In [9]:
rows = [list(MACCSkeys.GenMACCSKeys(mol)) for mol in mols]
n_bits = len(rows[0]) if rows else 167
maccs_df = pd.DataFrame(rows, columns=[f"maccs_{i}" for i in range(n_bits)])

maccs_df = pd.concat([meta[[SMILES_COL, LABEL_COL]], maccs_df], axis=1)
maccs_df.to_csv(OUTPUT_MACCS_CSV, index=False)

print(maccs_df.shape)
maccs_df.head()

(133885, 169)


,smiles,gap_eV,maccs_0,maccs_1,maccs_2,maccs_3,maccs_4,maccs_5,maccs_6,maccs_7,...,maccs_157,maccs_158,maccs_159,maccs_160,maccs_161,maccs_162,maccs_163,maccs_164,maccs_165,maccs_166
0,C,13.736308,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
1,N,9.249150,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,O,9.836916,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,C#C,9.118536,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C#N,10.329442,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


トレインテスト分割

In [12]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

INPUT_DIR = Path("input_data")
TRAIN_DIR = Path("train")
VAL_DIR = Path("val")
TEST_DIR = Path("test")

FILES = [
    "qm9_smiles_gap_eV.csv",
    "qm9_descriptors.csv",
    "qm9_morgan_fp.csv",
    "qm9_maccs_keys.csv",
]

TRAIN_SIZE = 0.8
VAL_SIZE = 0.1
TEST_SIZE = 0.1
RANDOM_STATE = 42
SMILES_COL = "smiles"

TRAIN_DIR.mkdir(exist_ok=True)
VAL_DIR.mkdir(exist_ok=True)
TEST_DIR.mkdir(exist_ok=True)

# 4ファイルを読み込む
dfs = {name: pd.read_csv(INPUT_DIR / name) for name in FILES}

# 4ファイルすべてに共通して存在するSMILESだけを対象にする
common_smiles = set(dfs[FILES[0]][SMILES_COL])
for name in FILES[1:]:
    common_smiles &= set(dfs[name][SMILES_COL])
common_smiles = sorted(common_smiles)  # 順序を固定して再現性を担保

print(f"共通のSMILES件数: {len(common_smiles)}")

# まずtestを切り出す（10%）
train_val_smiles, test_smiles = train_test_split(
    common_smiles, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# 残り(90%)をtrain(80%)とval(10%)に分割
val_ratio_within_remainder = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)  # 0.1 / 0.9
train_smiles, val_smiles = train_test_split(
    train_val_smiles, test_size=val_ratio_within_remainder, random_state=RANDOM_STATE
)

train_smiles_set = set(train_smiles)
val_smiles_set = set(val_smiles)
test_smiles_set = set(test_smiles)

print(f"train: {len(train_smiles)}件 / val: {len(val_smiles)}件 / test: {len(test_smiles)}件")

for name, df in dfs.items():
    df = df[df[SMILES_COL].isin(common_smiles)]
    df = df.drop_duplicates(subset=SMILES_COL, keep="first")  # 重複行を明示的に除去

    train_df = df[df[SMILES_COL].isin(train_smiles_set)].reset_index(drop=True)
    val_df = df[df[SMILES_COL].isin(val_smiles_set)].reset_index(drop=True)
    test_df = df[df[SMILES_COL].isin(test_smiles_set)].reset_index(drop=True)

    train_df.to_csv(TRAIN_DIR / name, index=False)
    val_df.to_csv(VAL_DIR / name, index=False)
    test_df.to_csv(TEST_DIR / name, index=False)

    print(f"{name}: train {train_df.shape} / val {val_df.shape} / test {test_df.shape}")

共通のSMILES件数: 133802
train: 107040件 / val: 13381件 / test: 13381件
qm9_smiles_gap_eV.csv: train (107040, 2) / val (13381, 2) / test (13381, 2)
qm9_descriptors.csv: train (107040, 219) / val (13381, 219) / test (13381, 219)
qm9_morgan_fp.csv: train (107040, 2050) / val (13381, 2050) / test (13381, 2050)
qm9_maccs_keys.csv: train (107040, 169) / val (13381, 169) / test (13381, 169)
